# 04 · From profiles to household presets

Notebook 03 produced **profiles**: for every item, the most common decision among a group of similar households. A
profile is 6,272 decisions, which nobody can read or edit. This notebook turns each profile into a **preset**: a short,
readable list of household rules (at most 15) that the app can show, edit and compile into PPD policies.

Steps:

1. **Fill the gaps.** Items where a profile has less than 80% agreement still need a decision, because the household
   is never asked (CableLabs guidance). A `FALLBACK` setting chooses how (notebook 03 compared the options).
2. **Compress** the 6,272 decisions into ≤ 15 rules with a small decision tree over the five fields. Each leaf of the
   tree is one rule: *"for these data types, purposes, … → deny / allow / allow if …"*. The leaves don't overlap, so
   exactly one rule applies to any item.
3. **Measure** what compression costs: fidelity to the profile, on all items and on the 52 real-device items, and how
   often the rules match each household's own decisions.
4. **Name** each preset from what sets it apart, and save the presets for notebooks 05 and 07.

**Rule format.** Each rule has a `scope` (lists of core terms per field; a missing field means "any") and an effect,
like the app's rules (`shared/types.ts`). PPD rules are allow or deny (protocol-04 §7.11); an *allow if …* rule is an
`allow` that carries a qualifier, which the compiler sends as `constraints` (§7.12). The app's rules cannot express
that condition yet: it needs a small extension (a `condition` on an allow rule).

In [1]:
import json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
items = pd.read_csv(DATA / "items.csv")
H = pd.read_csv(DATA / "households.csv")
D = np.load(DATA / "decisions.npy")
labels = pd.read_csv(DATA / "profiles" / "labels_ward.csv")
FIELDS = ["data_type", "purpose", "action", "source", "handling_context"]
TERMS = {f: sorted(items[f].unique()) for f in FIELDS}
CODES = {0: "deny", 1: "allow", 2: "allow if not kept", 3: "allow if in-home only", 4: "allow if on-device only"}
CONDITION = {2: "retention=ppd:ephemeral", 3: "processing_boundary=ppd:inHomeOnly", 4: "processing_boundary=ppd:onDeviceOnly"}
real = items.in_corpus.to_numpy()
home = ~items.leaves_home.to_numpy()
short = lambda term: term.split(":", 1)[1]
CANDIDATES = [int(c[1:]) for c in labels.columns if c.startswith("k")]
print("profiles available for k =", CANDIDATES)

profiles available for k = [4, 8, 12, 16, 20, 24]


## Settings

In [2]:
MAX_RULES = 15
THRESHOLD = 0.8
FALLBACK = "majority"   # for items below THRESHOLD: "majority" | "home_away" | "strict"
# How much each item counts when fitting the rules: real-device flows most, implausible combinations least.
WEIGHT_REAL, WEIGHT_PLAUSIBLE, WEIGHT_IMPLAUSIBLE = 5.0, 1.0, 0.1
weights = np.where(real, WEIGHT_REAL, np.where(items.plausible, WEIGHT_PLAUSIBLE, WEIGHT_IMPLAUSIBLE))

## Step 1 · Fill the gaps

- **majority**: use the profile's most common decision anyway. Highest accuracy in notebook 03.
- **home_away**: allow if the flow stays in the home, deny if it leaves. Cautious; like CableLabs' bulb firmware.
- **strict**: deny.

This is a privacy-versus-accuracy choice, not only a statistic: *majority* follows what similar households chose;
the cautious options deny some flows those households would have allowed.

In [3]:
def profile_decisions(k, fallback=FALLBACK):
    p = np.load(DATA / "profiles" / f"profiles_ward_k{k}.npz")
    majority, agreement = p["majority"], p["agreement"].astype(np.float32)
    if fallback == "majority":
        return majority, agreement
    fill = np.where(home, 1, 0) if fallback == "home_away" else np.zeros(len(items), dtype=np.uint8)
    return np.where(agreement >= THRESHOLD, majority, fill[None, :]).astype(np.uint8), agreement

## Step 2 · Compress into rules

A decision tree over the five fields (one yes/no feature per core term) predicts the profile's decision for each item,
with at most `MAX_RULES` leaves. Every condition on the path to a leaf restricts one field ("data type is content" or
"handling context is not the household"), so each leaf is a block of combinations: a list of allowed terms per field.
That is exactly a rule scope.

In [4]:
X = pd.get_dummies(items[FIELDS]).astype(np.float32)
FEATURE = [c.split("_ppd:")[0] and (c[: c.index("_ppd:")], "ppd:" + c.split("_ppd:")[1]) for c in X.columns]   # (field, term)
X = X.to_numpy()

def compress(decisions):
    # Fit the tree and turn each leaf into a rule: {scope, decision, items, fidelity}.
    tree = DecisionTreeClassifier(max_leaf_nodes=MAX_RULES, random_state=0).fit(X, decisions, sample_weight=weights)
    t = tree.tree_
    rules = []
    def walk(node, allowed):
        if t.children_left[node] == -1:   # a leaf
            decision = int(tree.classes_[t.value[node][0].argmax()])
            rules.append({"allowed": {f: sorted(v) for f, v in allowed.items()}, "decision": decision})
            return
        field, term = FEATURE[t.feature[node]]
        no = dict(allowed); no[field] = allowed[field] - {term}         # feature <= 0.5: the term is not used
        yes = dict(allowed); yes[field] = allowed[field] & {term}       # feature > 0.5: the term is used
        walk(t.children_left[node], no); walk(t.children_right[node], yes)
    walk(0, {f: set(TERMS[f]) for f in FIELDS})
    predicted = tree.predict(X).astype(np.uint8)
    for r in rules:
        inside = np.ones(len(items), dtype=bool)
        for f, terms in r["allowed"].items():
            inside &= items[f].isin(terms).to_numpy()
        r["items"] = int(inside.sum())
        r["fidelity"] = float((predicted[inside] == decisions[inside]).mean())
    return rules, predicted

### Rules in plain words

A field restricted to most of its terms reads better as "all except …".

In [5]:
FIELD_WORDS = {"data_type": "data", "purpose": "for", "action": "when", "source": "from", "handling_context": "handled by"}
def describe_terms(field, terms):
    everything = TERMS[field]
    if len(terms) == len(everything):
        return None
    missing = [t for t in everything if t not in terms]
    if len(missing) < len(terms):
        return f"{FIELD_WORDS[field]} anything except {', '.join(map(short, missing))}"
    return f"{FIELD_WORDS[field]} {', '.join(map(short, terms))}"

def rule_text(rule):
    parts = [p for f in FIELDS if (p := describe_terms(f, rule["allowed"][f]))]
    return f"{CODES[rule['decision']].upper()}: " + ("; ".join(parts) if parts else "everything else")

## Step 3 · Build the presets for every candidate k and measure them

In [6]:
def household_accuracy(k, per_item):
    lab = labels[f"k{k}"].to_numpy()
    return float(np.mean([(D[lab == c] == per_item[c]).mean() for c in range(k)]))

results, presets_by_k = [], {}
for k in CANDIDATES:
    decisions, agreement = profile_decisions(k)
    compressed = [compress(decisions[c]) for c in range(k)]
    per_item = np.stack([pred for _, pred in compressed])
    presets_by_k[k] = (decisions, agreement, compressed, per_item)
    results.append({
        "k": k,
        "rules per preset (max)": max(len(r) for r, _ in compressed),
        "fidelity to profile": float((per_item == decisions).mean()),
        "fidelity, real-device items": float((per_item[:, real] == decisions[:, real]).mean()),
        "household match: profile": household_accuracy(k, decisions),
        "household match: rules": household_accuracy(k, per_item),
    })
pd.DataFrame(results).set_index("k").round(3)

,rules per preset (max),fidelity to profile,"fidelity, real-device items",household match: profile,household match: rules
k,,,,,
4,15,0.882,0.947,0.751,0.719
8,15,0.887,0.957,0.771,0.739
12,15,0.890,0.952,0.770,0.742
16,15,0.891,0.957,0.776,0.748
20,15,0.892,0.957,0.776,0.749
24,15,0.890,0.955,0.777,0.748


Compression costs a few points of household match (the rules are coarser than the profile), but keeps each preset
to at most 15 readable rules. Real-device flows are fitted more closely because they carry more weight.

### Fallbacks, compared after compression (k = 8)

In [7]:
rows = {}
for fb in ["majority", "home_away", "strict"]:
    decisions, _ = profile_decisions(8, fb)
    per_item = np.stack([compress(decisions[c])[1] for c in range(8)])
    rows[fb] = {"household match": household_accuracy(8, per_item),
                "denied share": float((per_item == 0).mean()),
                "real-device flows denied": float((per_item[:, real] == 0).mean())}
pd.DataFrame(rows).T.round(3)

,household match,denied share,real-device flows denied
majority,0.739,0.287,0.175
home_away,0.621,0.566,0.296
strict,0.621,0.599,0.312


## Step 4 · Name the presets

A name is built from what sets each preset apart from all households: the purpose, recipient and data type where it
is most and least accepting. These are working names; people should rename them before they reach the app.

In [8]:
ordinal = np.array([0.0, 1.0, 0.5, 0.5, 0.5])
overall = ordinal[D].mean(axis=0)
def name_for(per_item):
    acc = ordinal[per_item]
    deltas = {}
    for f in ["purpose", "handling_context", "data_type"]:
        g = pd.Series(acc - overall).groupby(items[f].map(short)).mean()
        deltas.update({f"{f}:{k}": v for k, v in g.items()})
    s = pd.Series(deltas).sort_values()
    strict = [k.split(":")[1] for k in s.index[:2] if s[k] < -0.05]
    relaxed = [k.split(":")[1] for k in s.index[::-1][:2] if s[k] > 0.05]
    level = acc.mean()
    tone = "Strict" if level < 0.45 else "Careful" if level < 0.6 else "Relaxed" if level > 0.75 else "Balanced"
    bits = [tone] + ([f"no {', '.join(strict)}"] if strict else []) + ([f"OK with {', '.join(relaxed)}"] if relaxed else [])
    return " · ".join(bits)

## The presets at k = 8

Notebook 05 chooses k (more presets need more onboarding questions). k = 8 is shown here because it is readable.

In [9]:
K = 8
decisions, agreement, compressed, per_item = presets_by_k[K]
lab = labels[f"k{K}"].to_numpy()
overview = pd.DataFrame([{
    "preset": c, "name": name_for(per_item[c]), "households": int((lab == c).sum()),
    "rules": len(compressed[c][0]), "allowed share": float((per_item[c] == 1).mean()),
    "with conditions": float(np.isin(per_item[c], [2, 3, 4]).mean()), "denied": float((per_item[c] == 0).mean()),
    "fidelity": float((per_item[c] == decisions[c]).mean()),
} for c in range(K)]).set_index("preset")
overview.round(2)

,name,households,rules,allowed share,with conditions,denied,fidelity
preset,,,,,,,
0,"Careful · no thirdPartyContext, usageData · OK...",1062,15,0.45,0.09,0.46,0.89
1,"Balanced · no personalization, publicAudience ...",186,15,0.59,0.04,0.37,0.88
2,"Balanced · no thirdPartyContext, coreFunctiona...",732,15,0.62,0.07,0.31,0.88
3,Balanced · no analyticsAndImprovement · OK wit...,273,15,0.69,0.10,0.21,0.87
4,"Relaxed · OK with advertisingAndMarketing, ana...",957,15,0.79,0.11,0.10,0.88
5,Relaxed · no advertisingAndMarketing · OK with...,405,15,0.77,0.02,0.21,0.93
6,"Balanced · no analyticsAndImprovement, adverti...",228,15,0.63,0.08,0.29,0.89
7,"Balanced · no advertisingAndMarketing, publicA...",1157,15,0.59,0.06,0.35,0.87


In [10]:
for c in range(K):
    rules = sorted(compressed[c][0], key=lambda r: -r["items"])
    print(f"\n── Preset {c}: {overview.loc[c, 'name']}  ({overview.loc[c, 'households']} households, {len(rules)} rules)")
    for r in rules:
        print(f"   {rule_text(r)}   [{r['items']} items, {r['fidelity']:.0%} faithful]")


── Preset 0: Careful · no thirdPartyContext, usageData · OK with householdContext  (1062 households, 15 rules)
   DENY: data anything except deviceAndNetworkData; when anything except use; handled by publicAudience, thirdPartyContext   [2016 items, 89% faithful]
   ALLOW: handled by householdContext   [1568 items, 96% faithful]
   ALLOW IF IN-HOME ONLY: data anything except deviceAndNetworkData; for anything except advertisingAndMarketing; when use; handled by publicAudience, thirdPartyContext   [588 items, 79% faithful]
   DENY: data anything except deviceAndNetworkData; for anything except coreFunctionality, security; when inference, transfer; handled by vendorContext   [504 items, 82% faithful]
   ALLOW: data deviceAndNetworkData; for anything except advertisingAndMarketing, analyticsAndImprovement; when anything except use; handled by anything except householdContext   [378 items, 91% faithful]
   ALLOW: for anything except advertisingAndMarketing; when use; handled by vendorConte

## Checks

In [11]:
res = pd.DataFrame(results).set_index("k")
checks = {
    f"every preset has at most {MAX_RULES} rules": (res["rules per preset (max)"] <= MAX_RULES).all(),
    "rules reproduce most of each profile (>= 85%)": (res["fidelity to profile"] >= 0.85).all(),
    "real-device flows are reproduced closely (>= 93%)": (res["fidelity, real-device items"] >= 0.93).all(),
    "every item is decided by exactly one rule": all(sum(r["items"] for r in rules) == len(items)
                                                     for k in CANDIDATES for rules, _ in presets_by_k[k][2]),
    "presets differ from each other": len({tuple(p) for p in presets_by_k[K][3]}) == K,
}
assert all(checks.values()), checks
pd.Series(checks, name="holds")

every preset has at most 15 rules                    True
rules reproduce most of each profile (>= 85%)        True
real-device flows are reproduced closely (>= 93%)    True
every item is decided by exactly one rule            True
presets differ from each other                       True
Name: holds, dtype: bool

## Save

`data/presets/presets_k{k}.json` (rules in an app-like format) and `data/presets/preset_items_k{k}.npy` (the decision
each preset's rules give every item, for notebook 05).

In [12]:
APP_EFFECT = {0: "dont_allow", 1: "allow", 2: "allow", 3: "allow", 4: "allow"}
out = DATA / "presets"; out.mkdir(exist_ok=True)
for k in CANDIDATES:
    decisions, agreement, compressed, per_item = presets_by_k[k]
    lab = labels[f"k{k}"].to_numpy()
    presets = []
    for c in range(k):
        rules = []
        for i, r in enumerate(sorted(compressed[c][0], key=lambda r: -r["items"])):
            scope = {f: terms for f, terms in r["allowed"].items() if len(terms) < len(TERMS[f])}
            rules.append({"id": f"k{k}-p{c}-r{i}", "title": rule_text(r), "effect": APP_EFFECT[r["decision"]],
                          "condition": CONDITION.get(r["decision"]), "scope": scope, "enabled": True,
                          "items": r["items"], "fidelity": round(r["fidelity"], 3)})
        presets.append({"id": f"k{k}-p{c}", "name": name_for(per_item[c]), "households": int((lab == c).sum()),
                        "fidelity": round(float((per_item[c] == decisions[c]).mean()), 3), "rules": rules})
    (out / f"presets_k{k}.json").write_text(json.dumps({
        "k": k, "fallback": FALLBACK, "threshold": THRESHOLD, "max_rules": MAX_RULES,
        "note": "Synthetic presets (notebook 04). An 'allow' rule with a 'condition' means allow with that qualifier in constraints (protocol-04 7.11-7.12); the app needs a condition field to express it.",
        "presets": presets}, indent=2))
    np.save(out / f"preset_items_k{k}.npy", per_item)
print("saved", sorted(p.name for p in out.iterdir()))

saved ['preset_items_k12.npy', 'preset_items_k16.npy', 'preset_items_k20.npy', 'preset_items_k24.npy', 'preset_items_k4.npy', 'preset_items_k8.npy', 'presets_k12.json', 'presets_k16.json', 'presets_k20.json', 'presets_k24.json', 'presets_k4.json', 'presets_k8.json']
